# 🎮 R.M.M Make Roblox Mode (Roblox Mod & Experience AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/M.R.M-Make-Roblox-Mode/blob/main/R.M.M_Make_Roblox_Mode_Colab.ipynb)

> 자연어 요구사항만으로 **Roblox Studio 경험 및 인게임 모딩 시스템**을 자동 기획하고, Luau 코드를 정적 검증 및 Rojo 빌드한 뒤 **즉시 열 수 있는 `.rbxlx` 및 Rojo 프로젝트 ZIP**을 생성하는 올인원 Colab 노트북입니다.

- **Server engine gate**: 기본 활성화. 테스트용 Place에서 Roblox 엔진 검증이 통과해야 최종 ZIP 다운로드 (Studio 멀티플레이는 별도)
- **Full**: 기획 생성부터 Luau 생성, 정적 분석, 정적 검사 (실제 Studio 플레이테스트는 별도 필요), `.rbxlx` 및 Rojo ZIP 빌드까지 전체 실행
- **Plan**: 자연어 요구사항을 게임플레이, 네트워크(RemoteEvent), DataStore, UI 서브시스템으로 분해 및 기획안 생성
- **Revise**: 이전 계획 또는 체크포인트를 바탕으로 사용자 피드백을 반영하여 부분 수정
- **Execute**: 이미 확정된 `proposal.json` 계획 파일로 즉시 코드 생성 및 빌드 수행
- **Audit**: 프로젝트 전체 검사 및 Luau 린트 / 구문 분석 실행


In [ ]:
# @title 1. 실행 모드 및 게임 설정
import os

RUN_MODE = "Full" #@param ["Full", "Plan", "Revise", "Execute", "Audit"]
PROMPT = "코인 수집, 상점 시스템, 리더보드가 포함된 검투 아레나 게임을 만들어줘." #@param {type:"string"}
PLAN_FILE = "" #@param {type:"string"}
ROBLOX_EXPERIENCE_NAME = "CombatArena" #@param {type:"string"}
GENRE = "Action" #@param ["Action", "RPG", "Simulator", "Tycoon", "Obby", "Survival", "Minigame", "Custom"]
MODEL_PROFILE = "t4_local" #@param ["t4_local", "fast_test", "openai-compatible"]
REMOTE_BASE_URL = "" #@param {type:"string"}
REMOTE_API_KEY = "" #@param {type:"string"}
SEARCH_CREATOR_STORE = True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
ENGINE_VERIFICATION_POLICY = "Auto" #@param ["Auto", "Required", "Off"]
CLOUD_TEST_UNIVERSE_ID = "" #@param {type:"string"}
CLOUD_TEST_PLACE_ID = "" #@param {type:"string"}
CLOUD_ENABLE_TEST_PUBLISH = False #@param {type:"boolean"}
MAX_ENGINE_REBUILDS = 2 #@param {type:"integer"}

VALID_RUN_MODES = {"Full", "Plan", "Revise", "Execute", "Audit"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"지원하지 않는 실행 모드: {RUN_MODE}")

if RUN_MODE not in {"Execute", "Audit"} and not PROMPT.strip():
    raise ValueError("선택한 실행 모드에서는 PROMPT를 입력해야 합니다.")

os.environ["RMM_EXPERIENCE_NAME"] = ROBLOX_EXPERIENCE_NAME
os.environ["RMM_GENRE"] = GENRE
if CLOUD_TEST_UNIVERSE_ID.strip():
    os.environ["ROBLOX_TEST_UNIVERSE_ID"] = CLOUD_TEST_UNIVERSE_ID.strip()
else:
    os.environ.pop("ROBLOX_TEST_UNIVERSE_ID", None)
if CLOUD_TEST_PLACE_ID.strip():
    os.environ["ROBLOX_TEST_PLACE_ID"] = CLOUD_TEST_PLACE_ID.strip()
else:
    os.environ.pop("ROBLOX_TEST_PLACE_ID", None)
if CLOUD_ENABLE_TEST_PUBLISH:
    os.environ["RMM_ALLOW_TEST_PLACE_PUBLISH"] = "1"
else:
    os.environ.pop("RMM_ALLOW_TEST_PLACE_PUBLISH", None)
try:
    from google.colab import userdata
    cloud_key = userdata.get("ROBLOX_OPEN_CLOUD_API_KEY")
    if cloud_key:
        os.environ["ROBLOX_OPEN_CLOUD_API_KEY"] = cloud_key
except Exception:
    pass
# Auto performs every available verification; missing optional Roblox account
# credentials must NOT block code generation, offline repair or ZIP delivery.
# Required is opt-in strict remote-engine verification; Off is offline-only.
if ENGINE_VERIFICATION_POLICY not in {"Auto", "Required", "Off"}:
    raise ValueError("ENGINE_VERIFICATION_POLICY must be Auto, Required or Off")
CLOUD_MISSING_REQUIREMENTS = []
if not os.environ.get("ROBLOX_OPEN_CLOUD_API_KEY", "").strip():
    CLOUD_MISSING_REQUIREMENTS.append("ROBLOX_OPEN_CLOUD_API_KEY (Colab Secrets)")
for key in ("ROBLOX_TEST_UNIVERSE_ID", "ROBLOX_TEST_PLACE_ID"):
    value = os.environ.get(key, "").strip()
    if not value.isdecimal() or int(value or "0") <= 0:
        CLOUD_MISSING_REQUIREMENTS.append(key)
if os.environ.get("RMM_ALLOW_TEST_PLACE_PUBLISH") != "1":
    CLOUD_MISSING_REQUIREMENTS.append("CLOUD_ENABLE_TEST_PUBLISH=True")
CLOUD_ENGINE_READY = not CLOUD_MISSING_REQUIREMENTS
ENGINE_VERIFY_BEFORE_FINAL = (ENGINE_VERIFICATION_POLICY == "Required"
    or ENGINE_VERIFICATION_POLICY == "Auto" and CLOUD_ENGINE_READY)
if RUN_MODE in {"Full", "Execute", "Revise"} and not CLOUD_ENGINE_READY:
    if ENGINE_VERIFICATION_POLICY == "Required":
        raise RuntimeError(
            "Required 모드의 Roblox 서버 엔진 검증 설정이 부족합니다: "
            + ", ".join(CLOUD_MISSING_REQUIREMENTS)
            + ". 테스트용 Roblox Experience/Place를 설정하거나 Auto 모드를 사용하세요."
        )
    if ENGINE_VERIFICATION_POLICY == "Auto":
        print("[검증 자동 선택] Roblox 테스트 인증 정보가 없어 서버 엔진 검사는 건너뜁니다.")
        print("[진행] Luau 검사·수정·Rojo 빌드·ZIP 생성은 계속합니다.")
        print("[주의] 결과물은 built_unverified이며 Roblox 실플레이 검증 결과가 아닙니다.")
        print("[선택적 설정] " + ", ".join(CLOUD_MISSING_REQUIREMENTS))
if REMOTE_BASE_URL:
    os.environ["RMM_MODEL_BASE_URL"] = REMOTE_BASE_URL
    os.environ["ROBLOX_LLAMA_SERVER_URL"] = REMOTE_BASE_URL
if REMOTE_API_KEY:
    os.environ["RMM_MODEL_API_KEY"] = REMOTE_API_KEY


In [ ]:
# @title 2. GitHub 최신 main 설치 및 의존성 설정
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/M.R.M-Make-Roblox-Mode")
REPO_URL = "https://github.com/jujumelona/M.R.M-Make-Roblox-Mode.git"

IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "+main:refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
    elif REPO_DIR.exists():
        raise RuntimeError(f"Git 저장소가 아닌 경로가 이미 존재합니다: {REPO_DIR}")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

    USED_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    print("GitHub commit:", USED_COMMIT)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    # Colab keeps imported Python objects even when git reset updates the files.
    # An outdated LuauCodeGenerator plus fresh source linecache previously
    # produced mismatched design evidence and old weapon code after git sync.
    import importlib
    import linecache
    stale = [name for name in tuple(sys.modules)
             if name == "roblox_mod_ai" or name.startswith("roblox_mod_ai.")]
    for name in stale:
        sys.modules.pop(name, None)
    importlib.invalidate_caches()
    linecache.clearcache()
    if str(REPO_DIR) in sys.path:
        sys.path.remove(str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR))
    # Invalidate previously created session/proposal objects as well.
    for name in ("session", "FINAL_PLAN_REPLY", "BUILD_RESULT"):
        globals().pop(name, None)
    print("[sync] Invalidated", len(stale), "cached RMM modules")
    os.environ["RMM_SOURCE_COMMIT"] = USED_COMMIT
    import roblox_mod_ai
    assert Path(roblox_mod_ai.__file__).resolve().is_relative_to(REPO_DIR.resolve()), (
        "MRM was imported from a different installation; refusing mixed-version build")
    print("[sync] Imported repository:", roblox_mod_ai.__file__)
    OUTPUT_ROOT = Path("/content/rmm-output")
else:
    REPO_DIR = Path.cwd()
    OUTPUT_ROOT = REPO_DIR / "rmm-output"
    print("Local workspace detected:", REPO_DIR)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output directory initialized at: {OUTPUT_ROOT}")


# Install third-party CLI tools and start the model HERE, not in the status cell.
from roblox_mod_ai.colab_toolchain_bootstrap import setup_colab_toolchain
from roblox_mod_ai.colab_model_runtime import ensure_model_runtime

print("[setup] Installing Rojo, Wally, Selene and StyLua...", flush=True)
INSTALLED_TOOLCHAIN = setup_colab_toolchain(
    install_dir=OUTPUT_ROOT.parent / "rmm-tools" / "bin",
    in_colab=IN_COLAB,
)
print("[setup] Tools ready:", INSTALLED_TOOLCHAIN, flush=True)
os.environ["RMM_REQUIRE_REAL_LUAU_PARSER"] = "1"

print("[setup] Preparing real GGUF model...", flush=True)
MODEL_READY = None
if RUN_MODE != "Audit":
    MODEL_READY = ensure_model_runtime(
        profile=MODEL_PROFILE,
        remote_base_url=REMOTE_BASE_URL,
        api_key=REMOTE_API_KEY,
        cache_root=OUTPUT_ROOT.parent / "rmm-model",
        in_colab=IN_COLAB,
    )
    print("[setup] Model status:", MODEL_READY["status"], flush=True)
    print("[setup] Model:", MODEL_READY["model_id"], flush=True)
    print("[setup] API:", MODEL_READY["url"], flush=True)
print("[setup] Environment setup complete.", flush=True)


In [ ]:
# @title 3. 기존 입력 및 체크포인트 확인
import json
from pathlib import Path

EXISTING_PLAN_DATA = None
if RUN_MODE in {"Revise", "Execute"}:
    target_plan_path = None
    if PLAN_FILE.strip():
        target_plan_path = Path(PLAN_FILE).expanduser().resolve()
    else:
        default_candidate = OUTPUT_ROOT / "proposal.json"
        if default_candidate.is_file():
            target_plan_path = default_candidate

    if target_plan_path and target_plan_path.is_file():
        try:
            EXISTING_PLAN_DATA = json.loads(target_plan_path.read_text(encoding="utf-8"))
            print(f"기존 계획 파일 로드 성공: {target_plan_path}")
            print(f"  게임 제목: {EXISTING_PLAN_DATA.get('proposal', {}).get('title', 'Unknown')}")
            print(f"  서브시스템 수: {len(EXISTING_PLAN_DATA.get('proposal', {}).get('subsystems', []))}")
        except Exception as exc:
            print(f"계획 파일 로드 중 경고: {exc}")
    else:
        if RUN_MODE == "Execute":
            raise FileNotFoundError(f"Execute 모드에는 유효한 PLAN_FILE이 필요합니다: {target_plan_path}")
        print("기존 계획 파일이 없습니다. 새 세션으로 시작합니다.")
else:
    print(f"실행 모드: {RUN_MODE} (새 계획 생성)")


In [ ]:
# @title 4. Roblox API 카탈로그 및 도구체인 상태 확인
from roblox_mod_ai.creator_store import RobloxCreatorStoreCatalog
from roblox_mod_ai.roblox_api_catalog import RobloxAPICatalog
from roblox_mod_ai.roblox_toolchain import RobloxToolchainManager

api_catalog = RobloxAPICatalog()
services = api_catalog.list_services()
print(f"Roblox API Catalog 로드 완료: {len(services)}개 핵심 서비스 인덱싱됨")
print(f"  - 대표 서비스: {', '.join(services[:6])}...")

creator_catalog = RobloxCreatorStoreCatalog()
print("Creator Store searches use actual prompt intent; no forced asset queries")

toolchain = RobloxToolchainManager()
tool_status = toolchain.check_tools()
print("Roblox Toolchain 상태:")
for tool_name, available in tool_status.items():
    icon = "사용 가능" if available else "미설치 (필수 빌드는 실패)"
    print(f"  - {tool_name}: {icon}")



In [ ]:
# @title 5. 플랜 생성 및 서브시스템 분해
import json
from roblox_mod_ai.api import CompleteRobloxAISession

session = CompleteRobloxAISession(
    output_root=OUTPUT_ROOT,
    model_profile=MODEL_PROFILE,
    fast_mode=(MODEL_PROFILE == "fast_test"),
)

FINAL_PLAN_REPLY = None
if RUN_MODE == "Audit":
    print("Audit 모드: 계획 생성을 건너뜁니다.")
elif RUN_MODE == "Execute":
    if EXISTING_PLAN_DATA is not None:
        plan_file = OUTPUT_ROOT / "proposal.json"
        plan_file.write_text(json.dumps(EXISTING_PLAN_DATA, indent=2, ensure_ascii=False), encoding="utf-8")
        FINAL_PLAN_REPLY = session.load_plan(plan_file)
        print("기존 계획 로드 완료:")
        print(FINAL_PLAN_REPLY.message)
    else:
        raise ValueError("Execute 모드에 필요한 계획 데이터가 없습니다.")
elif RUN_MODE == "Revise":
    if EXISTING_PLAN_DATA is not None:
        plan_file = OUTPUT_ROOT / "proposal.json"
        plan_file.write_text(json.dumps(EXISTING_PLAN_DATA, indent=2, ensure_ascii=False), encoding="utf-8")
        session.load_plan(plan_file)
    else:
        session.plan(PROMPT)
    print(f"수정 피드백 반영 중: {PROMPT}")
    FINAL_PLAN_REPLY = session.revise(PROMPT)
    print(FINAL_PLAN_REPLY.message)
else:
    print(f"자연어 계획 생성 중: {PROMPT}")
    FINAL_PLAN_REPLY = session.plan(PROMPT)
    print("=" * 60)
    print(FINAL_PLAN_REPLY.message)
    print("=" * 60)

# Execute of an old, incomplete proposal must refresh its entire design,
# not repeatedly build the same semantic_decomposition_required checkpoint.
if RUN_MODE == "Execute" and FINAL_PLAN_REPLY is not None and not FINAL_PLAN_REPLY.ready_to_build:
    old_proposal = FINAL_PLAN_REPLY.authored_plan.proposal
    print("기존 계획은 미완성입니다. 원문 기준으로 다시 분석·설계합니다.")
    FINAL_PLAN_REPLY = session.plan(
        old_proposal.requested_prompt,
        project_mode=old_proposal.game_design.get("project_mode", "new_experience"),
    )

SAVED_PLAN_PATH = OUTPUT_ROOT / "proposal.json"
print(f"계획 저장 경로: {SAVED_PLAN_PATH}")


In [ ]:
# @title 6. Luau 코드 생성, 정적 분석 및 오프라인 Rojo 빌드
import json
from roblox_mod_ai.luau_analyzer import LuauStaticAnalyzer

BUILD_RESULT = None
CLOUD_RELEASE = None
PLAN_FAILURE_LOGS_ZIP = None

if RUN_MODE == "Plan":
    print("Plan 모드: 빌드를 건너뜁니다.")
elif RUN_MODE == "Audit":
    print("전체 프로젝트 및 Luau 스크립트 정적 감사(Audit) 실행 중...")
    analyzer = LuauStaticAnalyzer()
    src_dir = REPO_DIR / "src"
    total_files = 0
    issues_found = 0
    if src_dir.is_dir():
        for luau_file in src_dir.rglob("*.luau"):
            total_files += 1
            code = luau_file.read_text(encoding="utf-8")
            res = analyzer.analyze(code, filename=luau_file.name)
            if not res.valid or res.warnings:
                issues_found += len(res.errors) + len(res.warnings)
                print(f"  [{luau_file.name}] 오류: {len(res.errors)}, 경고: {len(res.warnings)}")
    print(f"Audit 완료: 총 {total_files}개 파일 검사, {issues_found}개 이슈 감지됨.")
else:
    if FINAL_PLAN_REPLY is None:
        raise RuntimeError("빌드할 계획이 없습니다. 5번 셀(플랜 준비)을 먼저 실행해 주세요.")

    if FINAL_PLAN_REPLY.ready_to_build:
        print("빌드 시작: 코드 생성 -> 정적 검사 및 에이전트 수정 -> Rojo 빌드")
        if ENGINE_VERIFY_BEFORE_FINAL:
            from roblox_mod_ai.verified_release import run_verified_release
            CLOUD_RELEASE = run_verified_release(
                session, FINAL_PLAN_REPLY, run_name=ROBLOX_EXPERIENCE_NAME,
                max_rebuilds=MAX_ENGINE_REBUILDS)
            print("실제 Roblox 서버 검증:", CLOUD_RELEASE["release_status"])
            print("최종 검증 ZIP:", CLOUD_RELEASE["final_zip"] or "없음")
            print("실패 진단 ZIP:", CLOUD_RELEASE["failure_report"] or "없음")
            print("Studio 멀티플레이 검증: 별도로 필요")
        else:
            BUILD_RESULT = session.build(FINAL_PLAN_REPLY, run_name=ROBLOX_EXPERIENCE_NAME)
        if BUILD_RESULT is not None:
            print("오프라인 빌드:", BUILD_RESULT.success)
            print("검증 상태:", BUILD_RESULT.metadata.get("release_status", "unverified"))
            if not BUILD_RESULT.success:
                print("실패 진단 ZIP:", BUILD_RESULT.metadata.get("failure_logs_zip", ""))
    else:
        PLAN_FAILURE_LOGS_ZIP = session.export_plan_failure()
        research = FINAL_PLAN_REPLY.authored_plan.proposal.game_design.get("pre_design_research", {})
        print("게임 설계가 검증되지 않았습니다. 생성 성공으로 표시하지 않습니다.")
        print("원인:", research.get("provider_failures", []))
        print("누락:", research.get("ledger", {}).get("unimplemented_facets", []))
        print("실패 로그:", PLAN_FAILURE_LOGS_ZIP)



In [ ]:
# @title 7. 결과물 다운로드 (.rbxlx 및 Rojo ZIP)
from pathlib import Path

DOWNLOAD_TARGETS = []

if CLOUD_RELEASE is not None:
    if CLOUD_RELEASE.get("verified"):
        DOWNLOAD_TARGETS.append(Path(CLOUD_RELEASE["final_zip"]))
        if CLOUD_RELEASE.get("place"):
            DOWNLOAD_TARGETS.append(Path(CLOUD_RELEASE["place"]))
    elif CLOUD_RELEASE.get("failure_report"):
        DOWNLOAD_TARGETS.append(Path(CLOUD_RELEASE["failure_report"]))
    if SAVED_PLAN_PATH.is_file():
        DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)
elif RUN_MODE == "Plan":
    if SAVED_PLAN_PATH.is_file():
        DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)
elif BUILD_RESULT is not None and BUILD_RESULT.success:
    if BUILD_RESULT.rbxlx_path and Path(BUILD_RESULT.rbxlx_path).is_file():
        DOWNLOAD_TARGETS.append(Path(BUILD_RESULT.rbxlx_path))
    if BUILD_RESULT.project_zip_path and Path(BUILD_RESULT.project_zip_path).is_file():
        DOWNLOAD_TARGETS.append(Path(BUILD_RESULT.project_zip_path))
    if SAVED_PLAN_PATH.is_file():
        DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)
elif SAVED_PLAN_PATH.is_file():
    DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)

if BUILD_RESULT is not None and not BUILD_RESULT.success:
    failure_path = BUILD_RESULT.metadata.get("failure_logs_zip", "")
    if failure_path and Path(failure_path).is_file():
        DOWNLOAD_TARGETS.append(Path(failure_path))
elif PLAN_FAILURE_LOGS_ZIP is not None and Path(PLAN_FAILURE_LOGS_ZIP).is_file():
    DOWNLOAD_TARGETS.append(Path(PLAN_FAILURE_LOGS_ZIP))

if not DOWNLOAD_TARGETS:
    print("다운로드할 결과 파일이 없습니다.")
else:
    print(f"총 {len(DOWNLOAD_TARGETS)}개의 결과물이 준비되었습니다:")
    for target in DOWNLOAD_TARGETS:
        print(f"  - {target.name} ({target.stat().st_size:,} bytes): {target}")

    try:
        from google.colab import files as colab_files
        for target in DOWNLOAD_TARGETS:
            print(f"Google Colab 다운로드 시작: {target.name}")
            colab_files.download(str(target))
    except (ImportError, Exception):
        print("로컬 환경에서는 위 파일 경로에서 직접 접근하거나 더블클릭할 수 있습니다.")


## Roblox Studio에서 결과물 실행

1. Colab은 코드 생성, 정적 검사와 Rojo 빌드를 수행합니다. 성공한 ZIP에는 compiled_place.rbxlx가 포함됩니다.
2. ZIP 압축을 풀고 compiled_place.rbxlx를 Roblox Studio에서 열어 Play로 게임을 확인합니다.
3. Windows에서 설치된 Roblox Studio를 사용해 로컬 구조 검사를 실행하려면 powershell -ExecutionPolicy Bypass -File .\local_studio\run_smoke.ps1 을 실행합니다. 이 검사는 실제 게임플레이나 멀티플레이 성공을 증명하지 않습니다.
4. 파일 실시간 동기화가 필요하면 Rojo Studio 플러그인을 설치하고 rojo serve를 실행한 뒤 Connect를 누릅니다.
5. AI로 Studio 플레이 테스트를 조작하려면 Studio의 내장 MCP 서버를 활성화하고 신뢰하는 로컬 MCP 클라이언트를 연결합니다. 예시 설정: local_studio/studio_mcp.windows.json.

실제 서버 엔진 검증은 테스트 전용 Place와 Universe ID, Colab Secrets의 ROBLOX_OPEN_CLOUD_API_KEY, 그리고 CLOUD_ENABLE_TEST_PUBLISH=True가 필요합니다. 정보가 없으면 최종 검증 ZIP 대신 진단을 제공합니다.
Colab은 사용자의 Windows PC에서 실행되는 Studio를 직접 제어하지 않습니다.
